In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import re
from pathlib import Path
from typing import Tuple

In [ ]:
import re
import math
import numpy as np
from pathlib import Path
from typing import Tuple

# Robust regex for scientific notation: 4.071099E+02
_RUNTIME_RE = re.compile(
    r"total\[s\]\s*=\s*([0-9]*\.?[0-9]+(?:E[+-]?\d+)?)",
    re.IGNORECASE,
)

# Assuming .gpu_peak file contains a number or "Max GPU (MiB) : 123"
_GPU_VAL_RE = re.compile(r"([0-9]*\.?[0-9]+)")

def read_run_stats(folder: str | Path, base_res: int, **kwargs) -> Tuple[float, float, float]:
    """
    Finds logs based on N{base_res}, extracts runtime and GPU memory.
    Returns (runtime_seconds, cpu_gb, gpu_gb). 
    Note: cpu_gb is returned as NaN as requested.
    """
    folder = Path(folder).expanduser().resolve()
    
    # 1. Locate the files
    timer_dir = folder / "timer_logs"
    perf_dir = folder / "logs_perf"
    
    # Find Timer Log: Matches Grid_rasBase_10_..._timer.log
    # We use glob with base_res. We look for rasBase_{N}_ to avoid N10 matching N100
    timer_pattern = f"Grid_rasBase_{base_res}_*timer.log"
    timer_files = list(timer_dir.glob(timer_pattern))
    
    # Find GPU Peak: Matches *N10*.gpu_peak
    gpu_pattern = f"*N{base_res}*.gpu_peak"
    gpu_files = list(perf_dir.glob(gpu_pattern))

    if not timer_files or not gpu_files:
        return (math.nan, math.nan, math.nan)

    # Pick the most recent if multiple exist
    timer_file = max(timer_files, key=lambda p: p.stat().st_mtime)
    gpu_file = max(gpu_files, key=lambda p: p.stat().st_mtime)

    # --- Extract Runtime ---
    try:
        timer_txt = timer_file.read_text(errors="replace")
        runtime_match = _RUNTIME_RE.search(timer_txt)
        runtime_seconds = float(runtime_match.group(1)) if runtime_match else math.nan
    except Exception:
        runtime_seconds = math.nan

    # --- Extract GPU Memory ---
    try:
        gpu_txt = gpu_file.read_text(errors="replace")
        # Find the last number in the file (often the peak)
        gpu_matches = _GPU_VAL_RE.findall(gpu_txt)
        if gpu_matches:
            # Assumes value is in MiB, convert to GB
            max_gpu_gb = float(gpu_matches[-1]) / 1024.0
        else:
            max_gpu_gb = math.nan
    except Exception:
        max_gpu_gb = math.nan

    return runtime_seconds, math.nan, max_gpu_gb

In [ ]:
# 1. Define your basic parameters
base_resolutions = [10, 13, 16, 20, 25, 31, 32, 35, 40]

# 2. Define the configurations (Folder, Label, Style)
# This replaces the 15+ separate lists you had before
configs = [
    {"folder": "cuda_res",  "label": "CUDA",    "marker": "x", "ls": "-"},
    {"folder": "n4_l2_res", "label": "n4 L2",   "marker": "s", "ls": "--"},
    {"folder": "n4_l3_res", "label": "n4 L3",   "marker": "o", "ls": "--"},
    {"folder": "n6_l2_res", "label": "n6 L2",   "marker": "^", "ls": ":"},
    {"folder": "n6_l3_res", "label": "n6 L3",   "marker": "d", "ls": ":"},
    {"folder": "n8_l2_res", "label": "n8 L2",   "marker": "v", "ls": "-."},
    {"folder": "n8_l3_res", "label": "n8 L3",   "marker": "<", "ls": "-."},
    {"folder": "n10_l2_res","label": "n10 L2",  "marker": ">", "ls": (0, (3, 1, 1, 1))},
    {"folder": "n10_l3_res","label": "n10 L3",  "marker": "*", "ls": (0, (3, 1, 1, 1))}
]

# 3. Gather the data
series_list = []

for cfg in configs:
    times = []
    gpus = []
    
    print(f"Reading data for {cfg['label']}...")
    
    for res in base_resolutions:
        # We only care about time and gpu_gb now
        t, _, gpu = read_run_stats(folder=cfg["folder"], base_res=res)
        times.append(t)
        gpus.append(gpu)
    
    # Bundle it up for the plotter
    series_list.append({
        "label": cfg["label"],
        "time": times,
        "gpu": gpus,
        "marker": cfg["marker"],
        "ls": cfg["ls"]
    })


In [ ]:
series_list[2]

In [ ]:
series_list[4]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_benchmark_results(
    base_resolutions,
    series_list,
    ignore_cpu_mem=True,
    fig_width_in=6.2,
    fig_height_in=3.8,
    font_size=9,
    label_size=10,
    tick_size=9,
    legend_size=8,
    line_width=1.4,
    marker_size=5.5,
    marker_edge_width=1.1,
):
    """
    series_list: List of dicts, e.g.:
    {
        "label": "CUDA",
        "time": [...],
        "cpu": [...],
        "gpu": [...],
        "marker": "x",
        "ls": "-"
    }
    """
    N = np.asarray(base_resolutions, dtype=float)
    x = N**3

    plt.rcParams.update({
        "font.size": font_size,
        "axes.labelsize": label_size,
        "axes.titlesize": label_size,
        "xtick.labelsize": tick_size,
        "ytick.labelsize": tick_size,
        "legend.fontsize": legend_size,
    })

    fig, ax_t = plt.subplots(figsize=(fig_width_in, fig_height_in))
    ax_m = ax_t.twinx()

    time_color = "tab:blue"
    cpu_color  = "tab:green"
    gpu_color  = "tab:red"

    common = dict(
        linewidth=line_width,
        markersize=marker_size,
        markeredgewidth=marker_edge_width,
    )

    def plot_nan_safe(ax, y, *, color, marker, linestyle, label):
        if y is None: return
        y = np.asarray(y, dtype=float)
        # Filter for valid data points
        m = np.isfinite(x) & np.isfinite(y) & (y > 0)
        if np.any(m):
            ax.plot(x[m], y[m], color=color, marker=marker, 
                    linestyle=linestyle, label=label, **common)

    # Iterate through each data series provided
    for s in series_list:
        label = s["label"]
        marker = s.get("marker", "o")
        ls = s.get("ls", "-")

        # Plot Time (Left Axis)
        plot_nan_safe(ax_t, s.get("time"), color=time_color, 
                      marker=marker, linestyle=ls, label=f"{label} time")
        
        # Plot CPU Memory (Right Axis)
        if not ignore_cpu_mem:
            plot_nan_safe(ax_m, s.get("cpu"), color=cpu_color, 
                          marker=marker, linestyle=ls, label=f"{label} CPU")
            
        # Plot GPU Memory (Right Axis)
        plot_nan_safe(ax_m, s.get("gpu"), color=gpu_color, 
                      marker=marker, linestyle=ls, label=f"{label} mem")

    # Log scales (Base 2 is common for cell-based resolutions)
    ax_t.set_xscale("log", base=2)
    ax_t.set_yscale("log", base=2)
    ax_m.set_yscale("log", base=2)

    ax_t.set_xlabel(r"Total number of cells ($N^3$)")
    ax_t.set_ylabel("Runtime [ s ]", color=time_color)
    ax_t.tick_params(axis="y", colors=time_color)
    
    ax_m.set_ylabel("Memory [ GB ]", color=gpu_color)
    ax_m.tick_params(axis="y", colors=gpu_color)

    # Legend handling
    h1, l1 = ax_t.get_legend_handles_labels()
    h2, l2 = ax_m.get_legend_handles_labels()
    if h1 or h2:
        ax_t.legend(h1 + h2, l1 + l2, loc="upper left", ncol=2, 
                    frameon=False, columnspacing=0.8, borderaxespad=0.2)

    ax_t.grid(True, which="both", linestyle=":", linewidth=0.8, alpha=0.6)
    fig.tight_layout(pad=0.2)
    plt.show()

In [ ]:
# 4. Final Plotting Call
plot_benchmark_results(
    base_resolutions=base_resolutions,
    series_list=series_list,
    ignore_cpu_mem=True,
    fig_width_in=3.45 * 1.8,  # Slightly wider for the legend
    fig_height_in=3.45 * 1.2
)

In [ ]:
# 2. Define the configurations (CUDA, n4, and n10)
configs = [
    {"folder": "cuda_res",  "label": "CUDA",    "marker": "x", "ls": "-"},
    {"folder": "n4_l2_res", "label": "n4 L2",   "marker": "s", "ls": "--"},
    {"folder": "n4_l3_res", "label": "n4 L3",   "marker": "o", "ls": "--"},
    {"folder": "n10_l2_res","label": "n10 L2",  "marker": ">", "ls": (0, (3, 1, 1, 1))},
    {"folder": "n10_l3_res","label": "n10 L3",  "marker": "*", "ls": (0, (3, 1, 1, 1))}
]

# 3. Gather the data
series_list = []

for cfg in configs:
    times = []
    gpus = []
    
    print(f"Reading data for {cfg['label']}...")
    
    for res in base_resolutions:
        t, _, gpu = read_run_stats(folder=cfg["folder"], base_res=res)
        times.append(t)
        gpus.append(gpu)
    
    series_list.append({
        "label": cfg["label"],
        "time": times,
        "gpu": gpus,
        "marker": cfg["marker"],
        "ls": cfg["ls"]
    })

import numpy as np
import matplotlib.pyplot as plt

def plot_benchmark_results(
    base_resolutions,
    series_list,
    ignore_cpu_mem=True,
    fig_width_in=6.2,
    fig_height_in=3.8,
    font_size=9,
    label_size=10,
    tick_size=9,
    legend_size=8,
    line_width=1.4,
    marker_size=5.5,
    marker_edge_width=1.1,
):
    N = np.asarray(base_resolutions, dtype=float)
    x = N**3

    plt.rcParams.update({
        "font.size": font_size,
        "axes.labelsize": label_size,
        "axes.titlesize": label_size,
        "xtick.labelsize": tick_size,
        "ytick.labelsize": tick_size,
        "legend.fontsize": legend_size,
    })

    fig, ax_t = plt.subplots(figsize=(fig_width_in, fig_height_in))
    ax_m = ax_t.twinx()

    time_color = "tab:blue"
    gpu_color  = "tab:red"

    common = dict(
        linewidth=line_width,
        markersize=marker_size,
        markeredgewidth=marker_edge_width,
    )

    def plot_nan_safe(ax, y, *, color, marker, linestyle, label):
        if y is None: return
        y = np.asarray(y, dtype=float)
        m = np.isfinite(x) & np.isfinite(y) & (y > 0)
        if np.any(m):
            ax.plot(x[m], y[m], color=color, marker=marker, 
                    linestyle=linestyle, label=label, **common)

    mem_plotted = set()

    # Iterate through each data series provided
    for s in series_list:
        label = s["label"]
        marker = s.get("marker", "o")
        ls = s.get("ls", "-")

        # Plot Time (Left Axis) - Exact label, no suffix
        plot_nan_safe(ax_t, s.get("time"), color=time_color, 
                      marker=marker, linestyle=ls, label=label)
        
        # Determine the memory grouping (CUDA, L2, or L3)
        if "CUDA" in label:
            level = "CUDA"
        elif "L2" in label:
            level = "L2"
        elif "L3" in label:
            level = "L3"
        else:
            level = None
        
        # Plot GPU Memory (Right Axis) - Only plot once per group
        if level and level not in mem_plotted:
            # Using a solid line for memory to keep it visually clean, 
            # while the marker shape and color distinguish the lines
            plot_nan_safe(ax_m, s.get("gpu"), color=gpu_color, 
                          marker=marker, linestyle="-", label=level)
            mem_plotted.add(level)

    # Log scales
    ax_t.set_xscale("log", base=2)
    ax_t.set_yscale("log", base=2)
    ax_m.set_yscale("log", base=2)

    ax_t.set_xlabel("Total number of cells ($N^3$)")
    ax_t.set_ylabel("Runtime [ s ]", color=time_color)
    ax_t.tick_params(axis="y", colors=time_color)
    
    ax_m.set_ylabel("Memory [ GB ]", color=gpu_color)
    ax_m.tick_params(axis="y", colors=gpu_color)

    # Combine legends seamlessly
    h1, l1 = ax_t.get_legend_handles_labels()
    h2, l2 = ax_m.get_legend_handles_labels()
    if h1 or h2:
        ax_t.legend(h1 + h2, l1 + l2, loc="upper left", ncol=2, 
                    frameon=False, columnspacing=0.8, borderaxespad=0.2)

    ax_t.grid(True, which="both", linestyle=":", linewidth=0.8, alpha=0.6)
    fig.tight_layout(pad=0.2)
    
    # Save as vector graphics for the poster
    fig.savefig("benchmark_results_poster.svg", format="svg", transparent=False, bbox_inches="tight")
    fig.savefig("benchmark_results_poster.pdf", format="pdf", transparent=False, bbox_inches="tight")
    
    plt.show()

# 4. Final Plotting Call
plot_benchmark_results(
    base_resolutions=base_resolutions,
    series_list=series_list,
    ignore_cpu_mem=True,
    fig_width_in=3.45 * 1.8,  
    fig_height_in=3.45 * 1.2
)